# DAQ session

Connect, configure, set the clock, acquire in a loop.

In [1]:
SIMULATED = True          # False, and PORT, at the bench
PORT = 'COM4'

In [2]:
from coaxial import Coaxial63100

device = Coaxial63100(port=PORT, simulated_device=SIMULATED).open()
print(device)

<Coaxial63100 Simulated SIMULATED>


In [3]:
daq = device.daq
daq.open()
for row in daq.catalogue():
    print('%-16s %-8s %-4s %-10s selectable=%s'
          % (row['name'], row['kind'], row['direction'], row['unit'],
             row['selectable']))

Phase U          analog   in   mA         selectable=True
Phase V          analog   in   mA         selectable=True
Phase W          analog   in   mA         selectable=True
Clevel           analog   in   None       selectable=True
NTC              analog   in   centi-degC selectable=True
DC bus           analog   in   mV         selectable=True
Cinj             analog   in   None       selectable=True
+5V              analog   in   mV         selectable=True
Vgate            analog   in   mV         selectable=True
MCU die          analog   in   centi-degC selectable=True
AFE_ON           digital  out  duty       selectable=True
UART5_TERM       digital  out  duty       selectable=True
KEEPALIVE        digital  out  duty       selectable=True
orientation      sensor   in   quaternion selectable=True
acceleration     sensor   in   m/s^2      selectable=True
rotation rate    sensor   in   rad/s      selectable=True
magnetic field   sensor   in   uT         selectable=True
shaft angle   

AFE_ON powers the ADC reference: with it off every channel reads exact mid-scale and the NTC exactly 25.00 C (invariant 9). `enable()` takes this session's reference on the rail; `close()` releases it.

In [4]:
daq.enable()
print(device.afe.state())

{'on': True, 'pe15': False, 'users': ['host']}


The board counts cycles, not time. `set_time_from_pc` ties the counter to the host's clock; `reference='utc'` measures that clock against NTP over the same window and takes out its offset and its rate, since a host clock is not a reference either.

In [5]:
sync = device.set_time_from_pc(reference='pc')
print(sync)

<Sync 474.994193 MHz (-12.2 ppm vs pc, floor 0.2 ppm), reference +/- 1 us>


In [6]:
layout = daq.configure('phaseU', 'NTC', sample_rate=50)
print(daq.channel_names())
print(layout)

['Phase U', 'NTC']
{'stride': 23, 'fields': [{'channel': 0, 'unit': 'mA', 'differential': True, 'signal': 'Phase U'}, {'channel': 4, 'unit': 'centi-degC', 'differential': False, 'signal': 'NTC'}], 'pins': [{'signal': 'AFE_ON', 'direction': 'out'}, {'signal': 'nFAULT/TIM1_BKIN', 'direction': 'in'}, {'signal': 'KEEPALIVE', 'direction': 'out'}, {'signal': 'TIM1_CH1N/PWMUL', 'direction': 'out'}, {'signal': 'TIM1_CH1/PWMUH', 'direction': 'out'}, {'signal': 'TIM1_CH2N/PWMVL', 'direction': 'out'}, {'signal': 'TIM1_CH2/PWMVH', 'direction': 'out'}, {'signal': 'TIM1_CH3N/PWMWL', 'direction': 'out'}, {'signal': 'TIM1_CH3/PWMWH', 'direction': 'out'}], 'sensors': []}


`start()` puts a reader thread on the link. Every `read(-1)` answers its own backlog: the first record blocks, the rest come with it.

In [7]:
import time

daq.start()
records = []
for _ in range(5):
    records.extend(daq.read(-1))
    time.sleep(0.2)
daq.stop()
for r in records[:8]:
    print('%.3f  dt %.4f  %s' % (r.start_time, r.dt,
                                 [(s.name, round(s.value, 1)) for s in r.samples]))
print('records:', len(records))
print(daq.state())
print(daq.buffered)

1788414007.455  dt 0.0200  [('Phase U', 3689.6), ('NTC', 38817.2)]
1788414007.475  dt 0.0200  [('Phase U', 3697.1), ('NTC', 38816.0)]
1788414007.495  dt 0.0200  [('Phase U', 3696.1), ('NTC', 38819.1)]
1788414007.515  dt 0.0200  [('Phase U', 3693.0), ('NTC', 38810.9)]
1788414007.535  dt 0.0200  [('Phase U', 3696.6), ('NTC', 38821.0)]
1788414007.555  dt 0.0200  [('Phase U', 3692.6), ('NTC', 38811.9)]
1788414007.575  dt 0.0200  [('Phase U', 3694.7), ('NTC', 38818.3)]
1788414007.595  dt 0.0200  [('Phase U', 3689.4), ('NTC', 38807.4)]
records: 340
{'running': False, 'done': False, 'lost_power': False, 'stride': 23, 'fields': 2, 'available': 0, 'produced': 420, 'dropped': 0, 'capacity': 19945, 'worst': 0, 'rung': 0, 'rungs': 0, 'rung_changes': 0, 'max_rate_hz': 375, 'sensors_available': 31, 'sensors_supported': True, 'channels': 17, 'clock': 'software', 'sample_time': 0, 'decimate': 1, 'accumulate': 0, 'records': 0, 'digital': True, 'interval_us': 20000, 'sensors': 0}
{'host': 0, 'peak': 0, 

A `Record` is a dict underneath: `r['NTC']` is the SUM over `r.count` readings, `r.value('NTC')` that channel's mean and `r.sample('NTC')` the struct behind it. `daq.series` and `daq.columns` are the two helpers around a whole run.

In [8]:
r = records[0]
print('sum   ', r['NTC'])
print('count ', r.count, ' (r["samples"] is the same number:', r['samples'], ')')
print('mean  ', r.value('NTC'))
print('sample', r.sample('NTC'))
print('names ', r.channel_name)
print('pins  ', r.digital)

sum    5123867.0
count  132  (r["samples"] is the same number: 132 )
mean   38817.17424242424
sample <NTC 38817.2 raw -> centi-degC>
names  ('Phase U', 'NTC')
pins   {'AFE_ON': 1.0, 'nFAULT/TIM1_BKIN': 1.0, 'KEEPALIVE': 0.4846230326035259, 'TIM1_CH1N/PWMUL': 0.0, 'TIM1_CH1/PWMUH': 0.0, 'TIM1_CH2N/PWMVL': 0.0, 'TIM1_CH2/PWMVH': 0.0, 'TIM1_CH3N/PWMWL': 0.0, 'TIM1_CH3/PWMWH': 0.0}


In [9]:
cols = daq.columns(records)
print(sorted(cols))
ntc = daq.series(records, 'NTC')
seconds = daq.series(records, 'time')
print('%.1f s of NTC, first %.1f last %.1f' % (seconds[-1] - seconds[0], ntc[0], ntc[-1]))

['AFE_ON', 'KEEPALIVE', 'NTC', 'Phase U', 'TIM1_CH1/PWMUH', 'TIM1_CH1N/PWMUL', 'TIM1_CH2/PWMVH', 'TIM1_CH2N/PWMVL', 'TIM1_CH3/PWMWH', 'TIM1_CH3N/PWMWL', 'dt', 'nFAULT/TIM1_BKIN', 'time']
-2.3 s of NTC, first 38817.2 last 37687.5


In [10]:
shape = daq.state()
held = daq.buffered
device.close()
print(device)

<Coaxial63100 Simulated SIMULATED>


## Conclusions

In [11]:
spans = [r.dt for r in records if r.dt]
counts = [r.count for r in records]
print('records          %d, %.2f s of covered time' % (len(records), sum(spans)))
print('record period    %.4f s mean = %.1f /s (asked for 50)'
      % (sum(spans) / len(spans), len(spans) / sum(spans)))
print('dt spread        %.4f to %.4f s' % (min(spans), max(spans)))
print('readings summed  %d to %d per record' % (min(counts), max(counts)))
print('stride           %d bytes, %d analog fields' % (shape['stride'], shape['fields']))
print('ring holds       %d records at this stride' % shape['capacity'])
print('dropped          %d, host queue peak %d' % (shape['dropped'], held['peak']))

records          340, 6.80 s of covered time
record period    0.0200 s mean = 50.0 /s (asked for 50)
dt spread        0.0200 to 0.0200 s
readings summed  132 to 132 per record
stride           23 bytes, 2 analog fields
ring holds       19945 records at this stride
dropped          0, host queue peak 0


`dt` is measured, not configured: it is the gap to the next record's timestamp within the block that carried it, because what the task was asked for and what the loop managed are different numbers - which is why the board sends a count with every sum. It is per block because the stamps are raw CYCCNT and that counter wraps every 9.04 s at 475 MHz; `_timed` unwraps each block it receives.

`dropped` is what the ring had no room for; a reader thread that keeps up leaves it at zero.